In [23]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File system
import os

In [24]:
# =========================
# IMPORTS
# =========================

import pandas as pd
import numpy as np

# =========================
# LOAD DATA
# =========================

products = pd.read_csv("../data/products.csv")
stores = pd.read_csv("../data/stores.csv")

# Verify they loaded correctly
print("Products:", products.shape)
print("Stores:", stores.shape)

products.head()

Products: (20, 6)
Stores: (10, 5)


,product_id,category,price,silhouette,fabric,trend_tags
0,1,Denim,205,Tailored,Cotton,Coastal Chic
1,2,Dress,332,Tailored,Nylon,Techwear
2,3,Cargo Pants,330,Relaxed,Silk,Coastal Chic
3,4,Dress,301,Relaxed,Nylon,Coastal Chic
4,5,Cargo Pants,174,Tailored,Wool,Coastal Chic


In [25]:
trend_weights = {
    "Quiet Luxury": 0.9,
    "Utility": 0.8,
    "Techwear": 0.7,
    "Coastal Chic": 0.75,
    "Power Dressing": 0.85
}

products["trend_score"] = products["trend_tags"].map(trend_weights)

In [26]:
def score_product_for_store(product, store):
    score = product["trend_score"]

    # Climate compatibility
    if store["climate"] == "Hot" and product["fabric"] in ["Silk","Nylon"]:
        score += 0.5
    if store["climate"] == "Cold" and product["fabric"] in ["Wool","Denim"]:
        score += 0.5

    # Price vs income
    if product["price"] < store["avg_income"] / 500:
        score += 0.4

    # Style match
    if product["category"] == "Blazer" and store["style_profile"] in ["Editorial","Luxury"]:
        score += 0.4

    return score

In [27]:
results = []

for _, p in products.iterrows():
    for _, s in stores.iterrows():
        score = score_product_for_store(p, s)
        results.append({
            "product_id": p["product_id"],
            "category": p["category"],
            "city": s["city"],
            "score": round(score,2)
        })

allocations = pd.DataFrame(results)

In [28]:
final_alloc = allocations.sort_values("score", ascending=False)\
                          .groupby("product_id")\
                          .head(3)

In [29]:
final_alloc.to_csv("../data/allocations.csv", index=False)

In [30]:
allocations.head()

,product_id,category,city,score
0,1,Denim,NYC,1.15
1,1,Denim,Miami,0.75
2,1,Denim,LA,1.15
3,1,Denim,Austin,0.75
4,1,Denim,Chicago,0.75
